### Credit Pricing Model with Machine Learning

Banks must factor in their own funding costs, the borrower’s default risk, profit margins, competitive landscape, and a long list of regulatory constraints.

* Loan pricing has long revolved around static cost-plus formulas:
* Funding cost — the interest banks pay to raise the capital (like deposit interest or wholesale borrowing).
* Operating cost — day-to-day administration and servicing expenses.
* Risk premium — the margin charged to offset expected losses from defaults.
* Profit margin — the target return on each loan, above and beyond costs.

Implement a two-stage pricing approach using Logistic Regression

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression

In [2]:
# 1. Simulate some data (for demo only)
n_samples = 1000
credit_score = np.random.randint(300, 851, size=n_samples)
competitor_rate = np.random.normal(0.05, 0.01, size=n_samples)
offered_rate = competitor_rate + np.random.uniform(0, 0.05, size=n_samples)

Historical Loan Data (offers, acceptances/rejections, default outcomes).

Funding & Capital Costs (updated monthly).

Competitor Rates (collected from market surveys).

Regulatory Rate Caps (e.g., no more than 24% APR).


Build an Acceptance Model (to predict who takes the loan at different rates) and a Default Risk Model (to predict PD and expected losses).

Deploy a simple algorithm that tests each possible rate — maybe from 6% to 20% — to see which maximizes expected profit.

Enforce a compliance check so final recommendations never exceed legal rate caps or go below cost of funds.

Acceptance model: assume acceptance depends on credit_score & difference from competitorm

In [3]:
logit_accept = 1.5 + (credit_score - 600)*(-0.002) + (competitor_rate - offered_rate)*50
prob_accept = 1 / (1 + np.exp(-logit_accept))
accepted = (np.random.rand(n_samples) < prob_accept).astype(int)

In [4]:
# Default model: assume default depends on credit_score & offered_rate
logit_default = -3.0 + (credit_score * -0.005) + (offered_rate * 20)
prob_default = 1 / (1 + np.exp(-logit_default))
defaulted = np.zeros(n_samples, dtype=int)
accepted_indices = np.where(accepted == 1)[0]
defaulted[accepted_indices] = (np.random.rand(len(accepted_indices)) < prob_default[accepted_indices]).astype(int)


In [5]:
# Train acceptance model
X_accept = np.column_stack((credit_score, competitor_rate, offered_rate))
y_accept = accepted
accept_model = LogisticRegression().fit(X_accept, y_accept)

In [6]:
# Train default model (only on accepted loans)
X_default = np.column_stack((credit_score[accepted_indices], offered_rate[accepted_indices]))
y_default = defaulted[accepted_indices]
default_model = LogisticRegression().fit(X_default, y_default)

In [7]:
# 2. Use models to find optimal rate for a new applicant
new_credit_score = 650
new_competitor_rate = 0.06
loan_amount = 10000
cost_of_funds = 0.04
capital_ratio = 0.1
cost_of_capital = 0.10
LGD = 0.5

rates = np.linspace(0.03, 0.15, 25)
best_rate, best_profit = None, -1

In [8]:
for r in rates:
    X_new_accept = [[new_credit_score, new_competitor_rate, r]]
    p_accept = accept_model.predict_proba(X_new_accept)[0, 1]

    X_new_def = [[new_credit_score, r]]
    p_default = default_model.predict_proba(X_new_def)[0, 1]

    revenue = r * loan_amount
    funding_cost_total = cost_of_funds * loan_amount
    capital_cost_total = capital_ratio * cost_of_capital * loan_amount
    expected_loss = p_default * LGD * loan_amount

    profit = p_accept * (revenue - funding_cost_total - capital_cost_total - expected_loss)
    if profit > best_profit:
        best_profit = profit
        best_rate = r

print(f"Optimal rate: {best_rate*100:.2f}%")
print(f"Expected profit: ${best_profit:.2f}")

Optimal rate: 15.00%
Expected profit: $459.50
